# 🎵 Đổi nền nhạc nhiều phong cách — Khúc ca Trường Kinh tế

Notebook chạy **miễn phí trên Google Colab** (bật GPU: *Runtime → Change runtime type → T4 GPU*).

Gồm 2 phần:
- **Phần A — MusicGen-Melody:** tạo **nhạc nền (instrumental)** bám theo giai điệu `score.mid`, nhiều phong cách.
- **Phần B — ACE-Step:** tạo **cả bài có giọng hát** (lời tiếng Việt) theo từng phong cách.

> ⚠️ Trước khi dùng, nên mở `score.mid` trong **MuseScore** để sửa nốt cho khớp bản gốc (OMR tự động không hoàn hảo).


## 0. Kiểm tra GPU


In [ ]:
!nvidia-smi -L || echo 'CHUA BAT GPU: Runtime -> Change runtime type -> T4 GPU'


## 1. Tải file melody `score.mid`
Chạy ô dưới rồi **chọn file `score.mid`** (lấy từ thư mục `song_arrangement/` của repo).


In [ ]:
from google.colab import files
up = files.upload()   # chon score.mid
MIDI_PATH = list(up.keys())[0]
print('Da tai:', MIDI_PATH)


---
# Phần A — MusicGen-Melody (nhạc nền bám melody)


### A1. Cài đặt


In [ ]:
!pip -q install audiocraft pretty_midi soundfile 2>/dev/null
print('done')


### A2. Render MIDI → audio melody
MusicGen-Melody cần **audio giai điệu mộc** làm điều kiện. Ta tổng hợp `score.mid` thành sóng sin (đơn âm, sạch).


In [ ]:
import pretty_midi, numpy as np, soundfile as sf
SR = 32000  # MusicGen dung 32kHz
pm = pretty_midi.PrettyMIDI(MIDI_PATH)
mel = pm.synthesize(fs=SR)            # tong hop song sin tu MIDI
mel = mel / (np.max(np.abs(mel))+1e-9)
sf.write('melody.wav', mel, SR)
dur = len(mel)/SR
print(f'Melody dai {dur:.1f}s')
from IPython.display import Audio; Audio('melody.wav')


### A3. Sinh nhạc nền theo 5 phong cách
MusicGen tạo tối đa ~30s/lần. Nếu bài dài hơn, model sẽ tự lặp/nối theo `duration` (đặt theo độ dài melody, tối đa 30s ở đây — chỉnh `DURATION` nếu cần).


In [ ]:
import torch, torchaudio
from audiocraft.models import MusicGen
from audiocraft.data.audio import audio_write

model = MusicGen.get_pretrained('facebook/musicgen-melody')  # 1.5B
DURATION = min(30, max(8, int(dur)))
model.set_generation_params(duration=DURATION)

STYLES = {
  'march':     'patriotic Vietnamese march, brass band, snare drum, steady 4/4, triumphant choir, proud and uplifting, orchestral',
  'ballad':    'emotional Vietnamese ballad, piano and strings, slow tempo, warm, heartfelt, gentle acoustic guitar, soft pad',
  'orchestra': 'epic cinematic orchestra, full strings, brass, timpani, grand and majestic, film score, soaring',
  'pop':       'modern Vietnamese pop, upbeat, bright synths, electric guitar, drums, energetic, youthful, catchy',
  'acoustic':  'acoustic guitar, light percussion, warm campfire vibe, organic, intimate, simple arrangement, claps',
}

# nap melody lam dieu kien
mel_wav, msr = torchaudio.load('melody.wav')
mel_wav = mel_wav[None]  # (B, C, T)

from IPython.display import Audio, display
for name, prompt in STYLES.items():
    print('==> Sinh phong cach:', name)
    wav = model.generate_with_chroma([prompt], mel_wav, msr)
    out = f'backing_{name}'
    audio_write(out, wav[0].cpu(), model.sample_rate, strategy='loudness')
    display(Audio(out + '.wav'))


### A4. Tải kết quả về máy


In [ ]:
from google.colab import files
import glob
for f in sorted(glob.glob('backing_*.wav')):
    files.download(f)


---
# Phần B — ACE-Step (cả bài: giọng hát + nhạc)

ACE-Step tạo bài hoàn chỉnh từ **lời + mô tả phong cách**. ⚠️ Tiếng Việt chưa đảm bảo phát âm chuẩn — hãy nghe thử; nếu cần chuẩn thì thu giọng thật và chỉ dùng Phần A cho nhạc nền.


### B1. Cài đặt ACE-Step


In [ ]:
!git clone https://github.com/ace-step/ACE-Step.git 2>/dev/null
%cd ACE-Step
!pip -q install -e . 2>/dev/null
%cd /content
print('done')


### B2. Lời bài hát (đã điền sẵn)


In [ ]:
LYRICS = '''[verse]
Bước trên con đường lòng hân hoan, ngàn ước mơ xanh dưới mái trường.
Những gian truân nhọc nhằn hôm qua để lại, cùng đắp xây tương lai bừng sáng.
Non sông đang đổi thay từng ngày, có chúng tôi chung bàn tay,
cùng mang yên vui đến cho mọi người, ấm áp trên môi nụ cười.
[chorus]
Cùng dựng xây đất nước phồn vinh muôn đời, những doanh nghiệp vươn ra thế giới.
Cùng điểm tô quê hương đẹp tươi muôn màu, và làm nên tổ quốc mạnh giàu.
Trường Kinh tế giữ sứ mệnh ươm nhân tài, tri thức luyện rèn cho tương lai,
vì cộng đồng sẽ chia giá trị, và chung tay vun đắp cuộc đời.
[outro]
Hát lên bạn ơi, khúc ca Trường Kinh tế,
chữ tín ta dựng xây bằng Chất lượng Thân thiện. Ôi tự hào Trường Kinh tế Đại học Cần Thơ.'''
print(LYRICS)


### B3. Sinh bài theo từng phong cách
Sửa `STYLE_TAGS` để đổi phong cách. ACE-Step dùng **tag mô tả** (genre, nhạc cụ, mood, tempo).


In [ ]:
from acestep.pipeline_ace_step import ACEStepPipeline
import torch
pipe = ACEStepPipeline(dtype='bfloat16' if torch.cuda.is_available() else 'float32')

STYLES = {
  'march':     'patriotic march, brass band, choir, proud, uplifting, 110 bpm',
  'ballad':    'emotional ballad, piano, strings, warm, slow, 70 bpm',
  'orchestra': 'epic cinematic orchestra, strings, brass, timpani, majestic, 90 bpm',
  'pop':       'vietnamese pop, upbeat, synth, electric guitar, drums, energetic, 120 bpm',
  'acoustic':  'acoustic guitar, light percussion, intimate, organic, 95 bpm',
}
AUDIO_DURATION = 90  # giay

from IPython.display import Audio, display
for name, tags in STYLES.items():
    print('==> ACE-Step phong cach:', name)
    out = f'song_{name}.wav'
    pipe(prompt=tags, lyrics=LYRICS, audio_duration=AUDIO_DURATION,
         infer_step=60, guidance_scale=15, save_path=out)
    display(Audio(out))


In [ ]:
from google.colab import files
import glob
for f in sorted(glob.glob('song_*.wav')):
    files.download(f)


---
### Ghi chú
- **Giữ đúng melody nhất:** Phần A (MusicGen-melody bám `score.mid`). Phần B tạo giai điệu mới theo lời, không bám `score.mid`.
- Muốn chính xác tuyệt đối: phối lại `score.mid` trong **MuseScore/DAW** với bộ nhạc cụ từng phong cách.
- API/tham số ACE-Step có thể đổi theo phiên bản — nếu lỗi, xem `ACE-Step/README` hoặc dùng giao diện Gradio: `!python ACE-Step/app.py`.
